# Bank Statement Anomaly Detection — Business Scoring Notebook (v0.1)

**Purpose (business track, no research/injection):** fit all detectors on the training CSV,
score the test CSV (last 4 months), flag anomalies per model, label each flagged row with an
anomaly type (volume_spike, large_amount, ...), and export a review file for analysts.

**How to use:**
1. Set `CFG.train_csv` and `CFG.test_csv` (same schema: 3 id columns + 17 features).
2. Set `CFG.predict_last_n_days` to limit how many recent test days are scored (applies to ALL models).
3. Run all cells. Analyst file lands in `./results_biz/`.
**Requirements:** pandas, numpy, scikit-learn, pyod, tqdm (pip install pyod tqdm). Optional: tabpfn, tabpfn-extensions.

4. TabPFN is optional (cell 7) — heavy on CPU; the day limit keeps it tractable.

**How a row gets its label:** a model flags a row when its score exceeds that model's threshold
(set at the `CFG.flag_percentile` of the model's own training scores). A flagged row is then typed
by comparing its features to the **account's own training history** (robust z-scores; global fallback
for accounts with < `min_account_rows` history). If no single feature stands out, the label is
`unusual_combination` — an honest "the pattern is odd but not one-dimensional" for the analyst.

In [ ]:
# ============ CONFIG ============
from dataclasses import dataclass

@dataclass
class Config:
    train_csv: str = "train.csv"
    test_csv: str = "test.csv"            # last 4 months, same schema
    results_dir: str = "results_biz"
    predict_last_n_days: int = 30         # score only the most recent N test days (all models)
    flag_percentile: float = 97.5         # per-model threshold on TRAIN scores (~2.5% flag rate)
    min_account_rows: int = 30            # per-account baseline needs this much history
    type_z_threshold: float = 3.0         # robust z needed to assign a named type
    active_models: tuple = ("Autoencoder(MLP)", "IForest")   # which detectors to run/report
    report_types: tuple = ("volume_spike",)   # only these anomaly types go to the analyst file; () = all
    run_tabpfn: bool = False              # flip True after pyod/AE pass works
    tabpfn_context_cap: int = 9800
    tabpfn_min_context: int = 50

CFG = Config()
import os; os.makedirs(CFG.results_dir, exist_ok=True)
print(CFG)

## 1. Library — schema, cleaning, type attribution

In [ ]:
"""Business-track library: load, clean, score, type-attribute. No injection."""
import numpy as np
import pandas as pd

ID_COLS = ["BankAccountCode", "txn_date", "CurrencyCode"]
FEATURE_COLS = [
    "txn_count", "amount_sum", "amount_mean", "amount_max", "amount_stddev",
    "credit_count", "debit_count", "credit_amount_sum", "debit_amount_sum",
    "unique_BU_count", "weekend_txn_count", "unique_txn_code_count",
    "top_bu_txn_count", "BU_concentration_ratio", "unique_currency_count",
    "credit_debit_ratio", "net_amount",
]
AMOUNT = ["amount_sum","amount_mean","amount_max","amount_stddev",
          "credit_amount_sum","debit_amount_sum"]

def validate_schema(df):
    issues = []
    missing = [c for c in ID_COLS + FEATURE_COLS if c not in df.columns]
    if missing: issues.append(f"Missing columns: {missing}")
    if not missing:
        if (df["txn_count"] <= 0).any(): issues.append("txn_count has non-positive values")
        bad = (df["credit_count"] + df["debit_count"] != df["txn_count"]).sum()
        if bad: issues.append(f"{bad} rows: credit_count+debit_count != txn_count")
    return issues

def clean_features(df):
    df = df.copy()
    df["txn_date"] = pd.to_datetime(df["txn_date"])
    df["BU_concentration_ratio"] = df["top_bu_txn_count"] / df["txn_count"]
    df["credit_debit_ratio"] = df["credit_count"] / (df["credit_count"] + df["debit_count"])
    for c in FEATURE_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    df[FEATURE_COLS] = df[FEATURE_COLS].replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return df

def build_context(day_df, history_df, cap=9800):
    accts = day_df["BankAccountCode"].unique()
    ctx = history_df[history_df["BankAccountCode"].isin(accts)]
    return ctx.sort_values("txn_date", ascending=False).head(cap)

# ---------------- Anomaly TYPE attribution ----------------
# robust z-score of each feature vs the account's own training history
# (global fallback when account has too few rows), mapped to anomaly types.

TYPE_RULES = [
    # (type, feature, direction)  direction +1 = anomalous when HIGH, -1 when LOW
    ("volume_spike",      "txn_count",              +1),
    ("large_amount",      "amount_max",             +1),
    ("large_amount",      "amount_sum",             +1),
    ("net_flow_extreme",  "net_amount_abs",         +1),
    ("bu_diversity",      "unique_BU_count",        +1),
    ("bu_concentration",  "BU_concentration_ratio", +1),
    ("code_diversity",    "unique_txn_code_count",  +1),
    ("weekend_activity",  "weekend_ratio",          +1),
    ("one_sided_flow",    "one_sidedness",          +1),
]

def _attr_frame(df):
    """Derived quantities used only for attribution."""
    a = pd.DataFrame(index=df.index)
    a["txn_count"] = df["txn_count"]
    a["amount_max"] = np.log1p(df["amount_max"].clip(lower=0))
    a["amount_sum"] = np.log1p(df["amount_sum"].clip(lower=0))
    a["net_amount_abs"] = np.log1p(df["net_amount"].abs())
    a["unique_BU_count"] = df["unique_BU_count"]
    a["BU_concentration_ratio"] = df["BU_concentration_ratio"]
    a["unique_txn_code_count"] = df["unique_txn_code_count"]
    a["weekend_ratio"] = df["weekend_txn_count"] / df["txn_count"].clip(lower=1)
    # 0 = balanced, 1 = all credits or all debits
    a["one_sidedness"] = (df["credit_debit_ratio"] - 0.5).abs() * 2
    return a

class TypeAttributor:
    def __init__(self, train_df, min_account_rows=30, z_flag=3.0):
        self.min_rows, self.z_flag = min_account_rows, z_flag
        self.attr_cols = None
        A = _attr_frame(train_df)
        self.attr_cols = list(A.columns)
        key = train_df["BankAccountCode"]
        med = A.groupby(key).median()
        mad = (A - med.reindex(key).set_index(A.index)).abs().groupby(key).median()
        cnt = A.groupby(key).size()
        ok = cnt[cnt >= min_account_rows].index
        self.acct_med, self.acct_mad = med.loc[ok], mad.loc[ok]
        self.glob_med = A.median()
        self.glob_mad = (A - self.glob_med).abs().median()
        # weekend habit per account: mean historical weekend_ratio (for the frequency guard)
        self.acct_wkend = A["weekend_ratio"].groupby(key).mean()

    def _z(self, row_attr, acct):
        if acct in self.acct_med.index:
            med, mad = self.acct_med.loc[acct], self.acct_mad.loc[acct]
            scope = "account"
        else:
            med, mad = self.glob_med, self.glob_mad
            scope = "global"
        # scale floor: account MAD, else a fraction of global MAD, else absolute floor.
        # prevents zero-variance features (e.g. weekend_ratio=0 all history) exploding z.
        scale = np.maximum(1.4826 * mad, np.maximum(0.25 * 1.4826 * self.glob_mad, 0.05))
        z = ((row_attr - med) / scale).clip(-50, 50)
        return z, scope

    def label(self, df):
        """Returns anomaly_type, top_feature, top_z, baseline_scope per row."""
        A = _attr_frame(df)
        out = []
        for i, acct in zip(df.index, df["BankAccountCode"]):
            z, scope = self._z(A.loc[i], acct)
            best_type, best_feat, best_z = "unusual_combination", None, 0.0
            wk_habit = float(self.acct_wkend.get(acct, 1.0))
            for t, feat, direction in TYPE_RULES:
                # weekend only counts as anomalous for accounts that almost never
                # transact on weekends (habit < 5% of activity)
                if t == "weekend_activity" and wk_habit >= 0.05:
                    continue
                zv = z[feat] * direction
                if zv >= self.z_flag and zv > best_z:
                    best_type, best_feat, best_z = t, feat, zv
            out.append((best_type, best_feat, round(float(best_z), 2), scope))
        return pd.DataFrame(out, index=df.index,
                            columns=["anomaly_type","top_feature","top_z","baseline_scope"])


## 2. Load train and test, apply the day limit

In [ ]:
import pandas as pd, numpy as np

train = pd.read_csv(CFG.train_csv)
test  = pd.read_csv(CFG.test_csv)
for name, d in [("train", train), ("test", test)]:
    iss = validate_schema(d)
    print(f"{name}: {len(d)} rows | validation:", iss if iss else "OK")
train, test = clean_features(train), clean_features(test)

# day limit (applies to every model)
days = sorted(test.txn_date.unique())
use_days = days[-CFG.predict_last_n_days:]
test = test[test.txn_date.isin(use_days)].reset_index(drop=True)
print(f"scoring {len(test)} test rows across {len(use_days)} days "
      f"({pd.Timestamp(use_days[0]).date()} to {pd.Timestamp(use_days[-1]).date()})")

## 3. Feature prep (fit on train only)

In [ ]:
from sklearn.preprocessing import StandardScaler

def prep(d):
    X = d[FEATURE_COLS].copy()
    for c in AMOUNT: X[c] = np.log1p(X[c].clip(lower=0))
    X["net_amount"] = np.sign(X["net_amount"]) * np.log1p(X["net_amount"].abs())
    return X

Ptr = prep(train)
KEEP = list(Ptr.columns[Ptr.std() > 1e-9])
print("features used:", len(KEEP), "| dropped zero-variance:", [c for c in Ptr.columns if c not in KEEP])
scaler = StandardScaler().fit(Ptr[KEEP])
def transform(d): return scaler.transform(prep(d)[KEEP])
Xtr, Xte = transform(train), transform(test)

## 4. Fit detectors, set per-model thresholds from train scores, score test

Threshold = `flag_percentile` of each model's own training-score distribution, mirroring
the production design (~2-3% flag rate).

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")
from sklearn.neural_network import MLPRegressor
from pyod.models.iforest import IForest
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.lof import LOF
from pyod.models.pca import PCA as PCAOD
from tqdm.auto import tqdm

class AEWrap:
    name = "Autoencoder(MLP)"
    def __init__(self): self.m = MLPRegressor(hidden_layer_sizes=(12,5,12), max_iter=400, random_state=0)
    def fit(self, X): self.m.fit(X, X); return self
    def score(self, X): return ((self.m.predict(X) - X)**2).mean(axis=1)

class PyODWrap:
    def __init__(self, name, model): self.name, self.m = name, model
    def fit(self, X): self.m.fit(X); return self
    def score(self, X): return np.nan_to_num(self.m.decision_function(X), nan=0.0, posinf=1e12, neginf=-1e12)

all_detectors = [AEWrap(),
                 PyODWrap("IForest", IForest(random_state=0)),
                 PyODWrap("ECOD", ECOD()),
                 PyODWrap("COPOD", COPOD()),
                 PyODWrap("LOF", LOF()),
                 PyODWrap("PCA", PCAOD())]
detectors = [d for d in all_detectors if d.name in CFG.active_models]
print("active detectors:", [d.name for d in detectors])

model_scores, model_thresholds = {}, {}
for det in tqdm(detectors, desc="Fit+score", unit="model"):
    det.fit(Xtr)
    thr = np.percentile(det.score(Xtr), CFG.flag_percentile)
    model_thresholds[det.name] = thr
    model_scores[det.name] = det.score(Xte)

print(pd.Series(model_thresholds).round(4).to_string())

## 5. Flag + type attribution

Every flagged row is labeled against the account's own history (global fallback for thin accounts).

In [ ]:
attributor = TypeAttributor(train, min_account_rows=CFG.min_account_rows,
                            z_flag=CFG.type_z_threshold)

long_rows = []
for mname, s in model_scores.items():
    thr = model_thresholds[mname]
    flagged_idx = test.index[s > thr]
    if len(flagged_idx):
        labels = attributor.label(test.loc[flagged_idx])
    for i in test.index:
        rec = dict(model=mname, row_id=int(i), score=float(s[i]),
                   threshold=float(thr), is_anomaly=bool(s[i] > thr))
        if rec["is_anomaly"]:
            rec.update(labels.loc[i].to_dict())
        long_rows.append(rec)

long_df = pd.DataFrame(long_rows)
flag_summary = long_df[long_df.is_anomaly].groupby("model").agg(
    flagged=("row_id","count")).join(
    long_df[long_df.is_anomaly].groupby("model")["anomaly_type"]
        .apply(lambda s: s.value_counts().to_dict()).rename("by_type"))
display(flag_summary)

## 6. Analyst review export

One row per (account, date, currency) that ANY model flagged, with per-model verdicts,
the consensus count, the assigned type, and the raw feature values the analyst needs to judge.

In [ ]:
flags = long_df[long_df.is_anomaly].copy()
if CFG.report_types:
    before = len(flags)
    flags = flags[flags.anomaly_type.isin(CFG.report_types)]
    print(f"type filter {CFG.report_types}: {before} flagged rows -> {len(flags)} reported")
stamp = pd.Timestamp.now().strftime("%Y%m%d")
path = f"{CFG.results_dir}/analyst_review_{stamp}.csv"
if len(flags) == 0:
    # write header-only file so the export always reflects the latest run (no stale files)
    cols = ID_COLS + FEATURE_COLS + ["n_models_flagged","anomaly_type","top_feature","max_top_z","baseline_scope"]
    pd.DataFrame(columns=cols).to_csv(path, index=False)
    long_df.to_csv(f"{CFG.results_dir}/all_scores_long_{stamp}.csv", index=False)
    print(f"No rows to report - none matched report_types (or none flagged). Empty file written: {path}")
    print("Widen CFG.report_types or lower flag_percentile if unexpected.")
else:
    pivot = flags.pivot_table(index="row_id", columns="model", values="score", aggfunc="first")
    pivot.columns = [f"score_{c}" for c in pivot.columns]
    consensus = flags.groupby("row_id").agg(
        n_models_flagged=("model","count"),
        anomaly_type=("anomaly_type", lambda s: s.mode().iat[0]),
        top_feature=("top_feature", lambda s: s.dropna().mode().iat[0] if s.dropna().size else None),
        max_top_z=("top_z","max"),
        baseline_scope=("baseline_scope","first"))
    out = (test.loc[consensus.index, ID_COLS + FEATURE_COLS]
           .join(consensus).join(pivot)
           .sort_values(["n_models_flagged","max_top_z"], ascending=False))
    out.to_csv(path, index=False)
    long_df.to_csv(f"{CFG.results_dir}/all_scores_long_{stamp}.csv", index=False)
    print(f"{len(out)} unique rows flagged by >=1 model -> {path}")
    print("consensus distribution (how many models agree):")
    print(out.n_models_flagged.value_counts().sort_index().to_string())
    display(out.head(10))

## 7. TabPFN (optional, per-account context — the production pattern)

Uses v2 weights for commercial-permissive licensing. Scores each test day against the history
of the accounts present that day. Honors the same day limit. Slow on CPU — the bar shows s/day.

In [ ]:
if CFG.run_tabpfn:
    try:
        from tabpfn import TabPFNClassifier, TabPFNRegressor
        from tabpfn.constants import ModelVersion
        from tabpfn_extensions import unsupervised as tab_unsup
        import torch

        def tabpfn_scores(ctx_X, test_X):
            clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
            reg = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
            m = tab_unsup.TabPFNUnsupervisedModel(clf, reg)
            m.fit(torch.as_tensor(np.asarray(ctx_X), dtype=torch.float32))
            s = m.outliers(torch.as_tensor(np.asarray(test_X), dtype=torch.float32))
            s = np.asarray(s, dtype=float).ravel()
            return np.nan_to_num(-s if np.nanmedian(s) > 0 else s, nan=0.0)  # higher = more anomalous; VERIFY sign

        # threshold from a train sample (cap for tractability)
        thr_sample = train.sort_values("txn_date").tail(CFG.tabpfn_context_cap)
        s_tr = tabpfn_scores(transform(thr_sample), transform(thr_sample))
        tab_thr = np.percentile(s_tr, CFG.flag_percentile)
        print("TabPFN threshold:", round(float(tab_thr), 4))

        scores_all = np.full(len(test), np.nan)
        for d in tqdm(sorted(test.txn_date.unique()), desc="TabPFN per-day", unit="day"):
            day_rows = test[test.txn_date == d]
            ctx = build_context(day_rows, train, cap=CFG.tabpfn_context_cap)
            if len(ctx) < CFG.tabpfn_min_context: continue
            scores_all[day_rows.index] = tabpfn_scores(transform(ctx), transform(day_rows))

        scored = ~np.isnan(scores_all)
        flagged_idx = test.index[scored & (scores_all > tab_thr)]
        labels = attributor.label(test.loc[flagged_idx]) if len(flagged_idx) else None
        tab_rows = []
        for i in test.index[scored]:
            rec = dict(model="TabPFN(per_account)", row_id=int(i), score=float(scores_all[i]),
                       threshold=float(tab_thr), is_anomaly=bool(scores_all[i] > tab_thr))
            if rec["is_anomaly"]: rec.update(labels.loc[i].to_dict())
            tab_rows.append(rec)
        long_df = pd.concat([long_df, pd.DataFrame(tab_rows)], ignore_index=True)
        print(f"TabPFN flagged {int((scores_all[scored] > tab_thr).sum())} of {int(scored.sum())} scored rows")
        print("Re-run cell 6 to regenerate the analyst export including TabPFN.")
    except Exception as e:
        print("TabPFN failed/unavailable:", e)
else:
    print("TabPFN disabled (CFG.run_tabpfn=False)")

## Notes for the analyst round
- `n_models_flagged` = consensus strength; start reviews from the top.
- `anomaly_type` is the modal label across flagging models; `unusual_combination` means multivariate-odd with no single dominant feature — these deserve attention, not dismissal.
- `baseline_scope` shows whether the type was judged against the account's own history or the global baseline (thin accounts).
- Collect analyst verdicts as a new column (`analyst_verdict`: confirmed / benign / unclear) in the exported CSV — that file becomes the seed labeled dataset for both tracks.